# 🚀 Autoencoders con Keras: comprimir, limpiar y detectar lo raro
*Curso de Machine Learning · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** entrenar una red que **comprime** una imagen y la **reconstruye**, y usarla para quitar ruido, ver el
espacio latente y detectar anomalías.

**Datos:** MNIST (dígitos escritos a mano).

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import tensorflow as tf
import keras
GPUS = tf.config.list_physical_devices("GPU")
print("Colab:", EN_COLAB, "| TensorFlow", tf.__version__, "| GPU:", GPUS[0].name if GPUS else "ninguna")
if not GPUS:
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")
keras.utils.set_random_seed(0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_test = x_train[..., None].astype("float32") / 255, x_test[..., None].astype("float32") / 255
if PRUEBA_RAPIDA:
    x_train, y_train, x_test, y_test = x_train[:3000], y_train[:3000], x_test[:500], y_test[:500]
EPOCAS = 1 if PRUEBA_RAPIDA else 10

def mostrar(filas, titulos):
    fig, ejes = plt.subplots(len(filas), 8, figsize=(10, 1.3 * len(filas)))
    for f, (fila, titulo) in enumerate(zip(filas, titulos)):
        for i in range(8):
            ejes[f, i].imshow(fila[i].squeeze(), cmap="gray"); ejes[f, i].axis("off")
        ejes[f, 0].set_title(titulo, fontsize=9, loc="left")
    plt.show()

## 1. El autoencoder denso

**Codificador:** 784 píxeles → 32 números (el *código*). **Decodificador:** 32 → 784. Se entrena para que la salida
se parezca a la entrada: la red aprende sola qué guardar en esos 32 números.

In [ ]:
codificador = keras.Sequential([keras.Input((28, 28, 1)), keras.layers.Flatten(),
                               keras.layers.Dense(128, activation="relu"), keras.layers.Dense(32, activation="relu")])
decodificador = keras.Sequential([keras.Input((32,)), keras.layers.Dense(128, activation="relu"),
                                 keras.layers.Dense(784, activation="sigmoid"), keras.layers.Reshape((28, 28, 1))])
autoencoder = keras.Sequential([codificador, decodificador])
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.fit(x_train, x_train, epochs=EPOCAS, batch_size=256, validation_data=(x_test, x_test), verbose=2)
mostrar([x_test, autoencoder.predict(x_test[:8], verbose=0)], ["original", "reconstruida desde 32 números"])

## 2. Quitar ruido: un autoencoder convolucional

In [ ]:
ruido = lambda x: np.clip(x + 0.4 * np.random.normal(size=x.shape), 0, 1).astype("float32")
x_train_ruido, x_test_ruido = ruido(x_train), ruido(x_test)

limpiador = keras.Sequential([
    keras.Input((28, 28, 1)),
    keras.layers.Conv2D(32, 3, activation="relu", padding="same"), keras.layers.MaxPooling2D(padding="same"),
    keras.layers.Conv2D(32, 3, activation="relu", padding="same"), keras.layers.MaxPooling2D(padding="same"),
    keras.layers.Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same"),
    keras.layers.Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same"),
    keras.layers.Conv2D(1, 3, activation="sigmoid", padding="same"),
])
limpiador.compile(optimizer="adam", loss="binary_crossentropy")
limpiador.fit(x_train_ruido, x_train, epochs=EPOCAS, batch_size=128, validation_data=(x_test_ruido, x_test), verbose=2)
mostrar([x_test_ruido, limpiador.predict(x_test_ruido[:8], verbose=0), x_test], ["con ruido", "limpiada", "original"])

## 3. El espacio latente en 2D

Con un código de solo **2 números** podemos dibujar dónde ubica la red cada dígito. Sin conocer las etiquetas, agrupa
los dígitos parecidos.

In [ ]:
cod2 = keras.Sequential([keras.Input((28, 28, 1)), keras.layers.Flatten(),
                        keras.layers.Dense(256, activation="relu"), keras.layers.Dense(2)])
dec2 = keras.Sequential([keras.Input((2,)), keras.layers.Dense(256, activation="relu"),
                        keras.layers.Dense(784, activation="sigmoid"), keras.layers.Reshape((28, 28, 1))])
ae2 = keras.Sequential([cod2, dec2]); ae2.compile(optimizer="adam", loss="binary_crossentropy")
ae2.fit(x_train, x_train, epochs=EPOCAS, batch_size=256, verbose=0)
z = cod2.predict(x_test, verbose=0)
plt.figure(figsize=(6, 5)); plt.scatter(z[:, 0], z[:, 1], c=y_test, cmap="tab10", s=4); plt.colorbar(label="dígito")
plt.title("Cada punto es un dígito comprimido a 2 números"); plt.show()

## 4. Detectar anomalías

Un autoencoder reconstruye bien **lo que se parece a lo que vio**. Si le muestras algo distinto, lo reconstruye mal:
un error de reconstrucción alto delata una anomalía.

In [ ]:
error = lambda x: np.mean((autoencoder.predict(x, verbose=0) - x) ** 2, axis=(1, 2, 3))
normales = error(x_test[:200])
raras = error(np.random.rand(200, 28, 28, 1).astype("float32"))          # ruido puro: nada parecido a un dígito
umbral = np.percentile(normales, 99)
print(f"Error típico de un dígito: {normales.mean():.4f} · de una imagen de ruido: {raras.mean():.4f}")
print(f"Con umbral {umbral:.4f}, se detectan como anómalas {np.mean(raras > umbral):.0%} de las imágenes de ruido")

## Pruébalo tú

Entrena el autoencoder denso con un código de **8** números en vez de 32 y guarda su pérdida de validación final en
`perdida_8`. ¿Cuánto empeora la reconstrucción al comprimir más?

In [ ]:
perdida_8 = None   # ← tu código

In [ ]:
# Comprobación
if perdida_8 is None:
    print("Aún no lo resuelves.")
elif perdida_8 > 0:
    print("✅ Registrada: compárala con la del código de 32")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- Un **autoencoder** aprende a comprimir y reconstruir sin etiquetas.
- Entrenado con entradas ruidosas y salidas limpias, **quita ruido**.
- El **error de reconstrucción** sirve para detectar anomalías.

**Fuentes:** [Keras · construir autoencoders (blog oficial)](https://blog.keras.io/building-autoencoders-in-keras.html) · [Keras · Conv2DTranspose](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/) · [Keras · MNIST](https://keras.io/api/datasets/mnist/)